<a href="https://colab.research.google.com/github/adityahadagalle/Machine_learning_projects/blob/main/IPL_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv("/content/deliveries.csv.zip")
df.head()

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN


In [57]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Step 1-2: Load Data
df = pd.read_csv("/content/deliveries.csv.zip")

# Step 4: Cleaning
consistent_teams = ['Kolkata Knight Riders', 'Chennai Super Kings', 'Rajasthan Royals',
                    'Mumbai Indians', 'Delhi Capitals', 'Royal Challengers Bangalore',
                    'Sunrisers Hyderabad', 'Punjab Kings', 'Gujarat Titans', 'Lucknow Super Giants']
team_mapping = {'Delhi Daredevils': 'Delhi Capitals', 'Kings XI Punjab': 'Punjab Kings', 'Deccan Chargers': 'Sunrisers Hyderabad'}
df['batting_team'] = df['batting_team'].replace(team_mapping)
df['bowling_team'] = df['bowling_team'].replace(team_mapping)
df = df[df['batting_team'].isin(consistent_teams) & df['bowling_team'].isin(consistent_teams)]

# Step 7: ULTIMATE Feature Engineering
total_score_df = df.groupby(['match_id', 'inning']).sum()['total_runs'].reset_index()
total_score_df.rename(columns={'total_runs': 'final_score'}, inplace=True)
df = df.merge(total_score_df, on=['match_id', 'inning'])

df['current_score'] = df.groupby(['match_id', 'inning'])['total_runs'].cumsum()
df['wickets'] = df.groupby(['match_id', 'inning'])['is_wicket'].cumsum()
df['overs'] = df['over'] + (df['ball'] / 6.0)

df['runs_last_5'] = df.groupby(['match_id', 'inning'])['total_runs'].transform(lambda x: x.rolling(window=30, min_periods=1).sum())
df['wickets_last_5'] = df.groupby(['match_id', 'inning'])['is_wicket'].transform(lambda x: x.rolling(window=30, min_periods=1).sum())
df['runs_last_3'] = df.groupby(['match_id', 'inning'])['total_runs'].transform(lambda x: x.rolling(window=18, min_periods=1).sum())
df['wickets_last_3'] = df.groupby(['match_id', 'inning'])['is_wicket'].transform(lambda x: x.rolling(window=18, min_periods=1).sum())
df['is_death_overs'] = (df['overs'] >= 15.0).astype(int)
df['batting_team_avg'] = df['batting_team'].map(df.groupby('batting_team')['final_score'].mean())
df['batting_team_avg']
df = df[df['overs'] >= 5.0]
final_df = df[['batting_team', 'bowling_team', 'overs', 'current_score', 'wickets', 'runs_last_5', 'wickets_last_5', 'runs_last_3', 'wickets_last_3', 'is_death_overs', 'batting_team_avg', 'final_score']]

# Step 5: Encoding
final_df = pd.get_dummies(data=final_df, columns=['batting_team', 'bowling_team'])

# Step 8-9: Split
X = final_df.drop(columns=['final_score'])
y = final_df['final_score']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 6: Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 10-13: ULTIMATE Model
model = ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)

# Step 11: Evaluation
y_pred = model.predict(X_test_scaled)
print(f"ULTIMATE R2 Score: {r2_score(y_test, y_pred):.2f}")
print(f"ULTIMATE RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")

def ultimate_predict(bat, bowl, ov, sc, wk, r5, w5, r3, w3):
    inp = pd.DataFrame(columns=X.columns).fillna(0)
    inp.loc[0]=0
    inp['overs'], inp['current_score'], inp['wickets'], inp['runs_last_5'], inp['wickets_last_5'], inp['runs_last_3'], inp['wickets_last_3'] = ov, sc, wk, r5, w5, r3, w3
    inp['is_death_overs'] = 1 if ov >= 15 else 0
    inp['batting_team_avg'] = df[df['batting_team']==bat]['batting_team_avg'].iloc[0] if bat in df['batting_team'].unique() else df['batting_team_avg'].mean()
    if f'batting_team_{bat}' in X.columns: inp[f'batting_team_{bat}'] = 1
    if f'bowling_team_{bowl}' in X.columns: inp[f'bowling_team_{bowl}'] = 1
    return int(model.predict(scaler.transform(inp))[0])

res = ultimate_predict('Mumbai Indians', 'Chennai Super Kings', 15.0, 130, 3, 50, 1, 35, 0)
print(f"Ultimate Prediction: {res} Runs")

ULTIMATE R2 Score: 0.91
ULTIMATE RMSE: 9.24
Ultimate Prediction: 178 Runs
